# Notebook 03: Vòng lặp Huấn luyện Độc lập, Checkpoint & Step Accounting (Phase D)

Notebook này kiểm chứng và minh họa kiến trúc huấn luyện của gói `dddcl`:
1. **Kiến trúc PyTorch MLP Classifier**: Khởi tạo trọng số deterministic, hỗ trợ Binary (`BCEWithLogitsLoss`) và Multiclass (`CrossEntropyLoss`).
2. **Tạo Mini-batches Ổn định**: Giữ nguyên thứ tự mẫu từ Curriculum mà không dùng random shuffle gây xáo trộn.
3. **Vòng lặp Huấn luyện SPL Từng Bước**:
   - Đánh giá lỗi $L_b$ của từng batch ở chế độ `eval()` không gradient.
   - Lọc batch Easy ($L_b < \lambda$) và Hard ($L_b \ge \lambda$).
   - Lượt 1: Huấn luyện trên batch Easy.
   - Kiểm tra điều kiện Replay (Algorithm 2) khi lỗi trung bình Easy $< \lambda$.
   - Cập nhật ngưỡng $\lambda$.
4. **Kế toán Điện toán (Accounting)**: Ghi nhận chính xác số bước `optimizer_steps` và số lượt tiếp xúc `sample_exposures`.
5. **Cơ chế Checkpoint Atomic & Resume**: Đảm bảo khôi phục bit-identical trạng thái mô hình và RNG.

In [1]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import tempfile

from dddcl.models.mlp import create_mlp_model
from dddcl.curriculum.geometry import compute_geometry
from dddcl.curriculum.partition import partition_by_class
from dddcl.curriculum.scoring import create_curriculum
from dddcl.curriculum.resample import resample_curriculum_bins
from dddcl.data.hf import load_dataset_from_release
from dddcl.data.splits import load_split
from dddcl.data.preprocess import prepare_processed_split
from dddcl.training.loop import (
    create_curriculum_batches,
    evaluate_batch_losses,
    evaluate_dataset,
    train_single_epoch_spl,
    train_dddcl_model,
)
from dddcl.training.checkpoint import save_checkpoint, load_checkpoint
from dddcl.runner import dry_run_matrix

print('Tất cả các module của dddcl đã sẵn sàng!')

Tất cả các module của dddcl đã sẵn sàng!


## 1. Khởi tạo Mô hình MLP & Kiểm tra Forward Pass

Mô hình `MLPClassifier` được thiết kế linh hoạt với seed khởi tạo deterministic, đầu ra trả về raw logits và tính toán loss phù hợp theo số lượng lớp.

In [2]:
# Kiểm tra mô hình binary (Haberman: 3 features, 2 classes)
mlp_binary = create_mlp_model(input_dim=3, num_classes=2, hidden_dims=(64, 32), seed=42)
x_dummy = torch.randn(4, 3)
logits_bin = mlp_binary(x_dummy)
probs_bin = mlp_binary.predict_proba(x_dummy)
preds_bin = mlp_binary.predict(x_dummy)

print('Binary Model Architecture:')
print(mlp_binary)
print('\nLogits shape:', logits_bin.shape)  # (4, 1)
print('Probabilities shape:', probs_bin.shape)  # (4, 2)
print('Predictions:', preds_bin)

Binary Model Architecture:
MLPClassifier(
  (network): Sequential(
    (0): Linear(in_features=3, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

Logits shape: torch.Size([4, 1])
Probabilities shape: (4, 2)
Predictions: [0 0 0 0]


## 2. Nạp Dữ liệu Pilot & Xây dựng Mini-batches theo Curriculum

Chúng ta nạp tập dữ liệu `haberman` (đã xác thực từ `hf_release`), nạp file split phân tầng seed 0, chuẩn hóa dữ liệu trên train-only, tính toán thứ tự curriculum (`sp_ddcl_point`) và tạo các mini-batches ổn định.

In [3]:
release_dir = Path('../hf_release') if Path('../hf_release').exists() else Path('hf_release')
ds_bundle = load_dataset_from_release(release_dir, 'haberman')
splits_dir = Path('../outputs/splits') if Path('../outputs/splits').exists() else Path('outputs/splits')
split_file = splits_dir / 'paper_interpretation_clean_v1' / 'haberman' / 'seed_0.json'
split_bundle = load_split(split_file, bundle=ds_bundle)

# Tiền xử lý (StandardScaler fit trên train)
processed = prepare_processed_split(ds_bundle, split_bundle)

# Tính curriculum point
geom = compute_geometry(processed.X_train_raw, processed.y_train)
part = partition_by_class(geom.normalized_distances, processed.y_train, n_levels=5)
curr_res = create_curriculum(
    strategy='sp_ddcl_point',
    sample_ids=processed.train_ids,
    normalized_distances=geom.normalized_distances,
    bin_assignments=part.bin_assignments,
    seed=0,
)

# Tạo các batches kích thước 32
batches = create_curriculum_batches(
    X_train=processed.X_train_scaled,
    y_train=processed.y_train,
    train_ids=processed.train_ids,
    curriculum_result=curr_res,
    batch_size=32,
)

print(f'Tổng số mẫu train: {len(processed.train_ids)}')
print(f'Số lượng mini-batches: {len(batches)}')
for b in batches:
    print(f'  Batch {b.batch_id}: {len(b.sample_ids)} mẫu, X shape: {b.X.shape}')

Tổng số mẫu train: 213
Số lượng mini-batches: 7
  Batch 0: 32 mẫu, X shape: torch.Size([32, 3])
  Batch 1: 32 mẫu, X shape: torch.Size([32, 3])
  Batch 2: 32 mẫu, X shape: torch.Size([32, 3])
  Batch 3: 32 mẫu, X shape: torch.Size([32, 3])
  Batch 4: 32 mẫu, X shape: torch.Size([32, 3])
  Batch 5: 32 mẫu, X shape: torch.Size([32, 3])
  Batch 6: 21 mẫu, X shape: torch.Size([21, 3])


## 3. Thực thi Vòng lặp SPL 1 Epoch và Kiểm tra Kế toán Chi tiết

Theo dõi cơ chế tách batch Easy/Hard ở đầu epoch, cập nhật trọng số trên Easy batches, và kiểm tra điều kiện Replay.

In [4]:
model = create_mlp_model(input_dim=processed.X_train_scaled.shape[1], num_classes=2, seed=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Đánh giá losses trước update
initial_losses = evaluate_batch_losses(model, batches)
print('Batch losses ban đầu khi chưa huấn luyện:', [f'{l:.4f}' for l in initial_losses])

# Chạy 1 epoch SPL với ngưỡng lambda = 0.4
epoch_acct, next_lambda = train_single_epoch_spl(
    model=model,
    optimizer=optimizer,
    batches=batches,
    current_lambda=0.4,
    pace_policy='replay_easy_then_union_v1',
)

print(f'\nKết quả Epoch 1 SPL:')
print(f'  - Ngưỡng lambda sử dụng: {epoch_acct.lambda_value}')
print(f'  - Các batch Easy: {epoch_acct.easy_batch_ids}')
print(f'  - Các batch Hard: {epoch_acct.hard_batch_ids}')
print(f'  - Có kích hoạt Replay không? {epoch_acct.replayed}')
print(f'  - Số bước optimizer thực hiện: {epoch_acct.optimizer_steps}')
print(f'  - Số lượt mẫu tiếp xúc: {epoch_acct.sample_exposures}')
print(f'  - Thời gian thực thi: {epoch_acct.duration_seconds:.4f}s')

Batch losses ban đầu khi chưa huấn luyện: ['0.7274', '0.8509', '0.6025', '0.5742', '0.6711', '0.7458', '1.6927']

Kết quả Epoch 1 SPL:
  - Ngưỡng lambda sử dụng: 0.4
  - Các batch Easy: [3]
  - Các batch Hard: [2, 4, 0, 5, 1, 6]
  - Có kích hoạt Replay không? False
  - Số bước optimizer thực hiện: 1
  - Số lượt mẫu tiếp xúc: 32
  - Thời gian thực thi: 0.0062s


## 4. Xác minh Tính Tái lập & Khôi phục Trạng thái (Bit-identical Checkpoint Resume)

Chúng ta thực hiện kiểm thử thực nghiệm: huấn luyện 4 epochs liên tục so sánh với việc dừng ở epoch 2, lưu checkpoint, khởi động lại và resume 2 epochs tiếp theo.

In [5]:
# Chạy 4 epochs liên tục
torch.manual_seed(99)
m_unint = create_mlp_model(input_dim=3, num_classes=2, seed=99)
opt_unint = torch.optim.Adam(m_unint.parameters(), lr=0.005)
_, hist_unint = train_dddcl_model(
    model=m_unint,
    optimizer=opt_unint,
    batches=batches,
    strategy='sp_ddcl_point',
    epochs=4,
    lambda_init=0.4,
    lambda_max=0.6,
    lambda_growth=1.1,
    run_fingerprint='walkthrough_demo',
)

# Chạy 2 epochs + Checkpoint
with tempfile.TemporaryDirectory() as tmpdir:
    ckpt_dir = Path(tmpdir)
    torch.manual_seed(99)
    m_res = create_mlp_model(input_dim=3, num_classes=2, seed=99)
    opt_res = torch.optim.Adam(m_res.parameters(), lr=0.005)
    train_dddcl_model(
        model=m_res,
        optimizer=opt_res,
        batches=batches,
        strategy='sp_ddcl_point',
        epochs=2,
        lambda_init=0.4,
        lambda_max=0.6,
        lambda_growth=1.1,
        run_fingerprint='walkthrough_demo',
        checkpoint_dir=ckpt_dir,
        checkpoint_freq=1,
    )
    
    # Resume 2 epochs tiếp theo
    _, hist_res = train_dddcl_model(
        model=m_res,
        optimizer=opt_res,
        batches=batches,
        strategy='sp_ddcl_point',
        epochs=4,
        lambda_init=0.4,
        lambda_max=0.6,
        lambda_growth=1.1,
        run_fingerprint='walkthrough_demo',
        checkpoint_dir=ckpt_dir,
        resume=True,
    )

# Kiểm tra bit-identical
weights_match = all(
    torch.equal(m_unint.state_dict()[k], m_res.state_dict()[k])
    for k in m_unint.state_dict()
)
loss_unint = [h['train_loss'] for h in hist_unint]
loss_res = [h['train_loss'] for h in hist_res]

print('Loss 4 epochs liên tục:', [f'{l:.6f}' for l in loss_unint])
print('Loss 2 + resume 2 epochs:', [f'{l:.6f}' for l in loss_res])
print(f'Mô hình weights giống nhau 100% bit-identical: {weights_match}')
assert weights_match, 'Resume weights mismatch!'

Loss 4 epochs liên tục: ['0.611013', '0.494819', '0.378455', '0.296939']
Loss 2 + resume 2 epochs: ['0.611013', '0.494819', '0.378455', '0.296939']
Mô hình weights giống nhau 100% bit-identical: True


## 5. Kiểm tra Kế hoạch Thực nghiệm Matrix (--dry-run)

Gọi hàm `dry_run_matrix` để kiểm tra toàn bộ 54 runs của cấu hình pilot `configs/pilot.yaml` mà không thực hiện huấn luyện hay sửa đổi dữ liệu.

In [6]:
cfg_path = Path('../configs/pilot.yaml') if Path('../configs/pilot.yaml').exists() else Path('configs/pilot.yaml')
plan = dry_run_matrix(cfg_path)

df_runs = pd.DataFrame(plan['runs'])
print(f'Config: {plan["config_file"]}')
print(f'Tổng số runs: {plan["total_fits"]}')
print(f'Số lượng dataset: {len(plan["datasets"])} {plan["datasets"]}')
print(f'Số lượng strategy: {len(plan["strategies"])} {plan["strategies"]}')
print(f'Số lượng seed: {len(plan["seeds"])} {plan["seeds"]}\n')
print('Trích đoạn ma trận thực nghiệm:')
display(df_runs.head(12))

Config: ..\configs\pilot.yaml
Tổng số runs: 54
Số lượng dataset: 3 ['haberman', 'new_thyroid', 'banknote']
Số lượng strategy: 6 ['no_curriculum', 'ddcl_density', 'ddcl_point', 'self_paced', 'sp_ddcl_density', 'sp_ddcl_point']
Số lượng seed: 3 [0, 1, 2]

Trích đoạn ma trận thực nghiệm:


,run_id,dataset,strategy,seed,completed
0,haberman_no_curriculum_s0,haberman,no_curriculum,0,False
1,haberman_no_curriculum_s1,haberman,no_curriculum,1,False
2,haberman_no_curriculum_s2,haberman,no_curriculum,2,False
3,haberman_ddcl_density_s0,haberman,ddcl_density,0,False
4,haberman_ddcl_density_s1,haberman,ddcl_density,1,False
5,haberman_ddcl_density_s2,haberman,ddcl_density,2,False
6,haberman_ddcl_point_s0,haberman,ddcl_point,0,False
7,haberman_ddcl_point_s1,haberman,ddcl_point,1,False
8,haberman_ddcl_point_s2,haberman,ddcl_point,2,False
9,haberman_self_paced_s0,haberman,self_paced,0,False
